In [1]:
import os
from dotenv import load_dotenv
load_dotenv()
from hdbcli import dbapi

# HANA Credentials
address = os.getenv("HANA_HOST")
port = os.getenv("HANA_PORT")
user = os.getenv("HANA_USER")
password = os.getenv("HANA_PASSWORD")

connection = dbapi.connect(address=address, port=port, user=user, password=password)
cursor = connection.cursor()

pbf_query = """
SELECT PBF_NAME, IMPORT_STATUS, IMPORT_TS FROM "GEO_FAB"."PBF_IMPORTS";
"""

cursor.execute(pbf_query)
pbf_rows = cursor.fetchall()

cursor.close()
connection.close()

import pandas as pd
pbf_df = pd.DataFrame(pbf_rows, columns=['PBF_NAME', 'IMPORT_STATUS', 'IMPORT_TS'])

pbf_df.to_dict(orient='records')

[{'PBF_NAME': 'andorra-latest.osm.pbf',
  'IMPORT_STATUS': 'FAILED',
  'IMPORT_TS': Timestamp('2025-12-03 10:59:08.238218')},
 {'PBF_NAME': 'isle-of-man-251117.osm.pbf',
  'IMPORT_STATUS': 'FAILED',
  'IMPORT_TS': Timestamp('2025-12-03 11:00:29.752237')},
 {'PBF_NAME': 'latvia-latest.osm.pbf',
  'IMPORT_STATUS': 'FAILED',
  'IMPORT_TS': Timestamp('2025-12-03 11:31:56.206416')},
 {'PBF_NAME': 'albania-251117.osm.pbf',
  'IMPORT_STATUS': 'IN_PROGRESS',
  'IMPORT_TS': Timestamp('2025-12-03 12:42:21.821054')},
 {'PBF_NAME': 'estonia-251127.osm.pbf',
  'IMPORT_STATUS': 'FAILED',
  'IMPORT_TS': Timestamp('2025-12-03 13:01:19.085699')},
 {'PBF_NAME': 'belarus-latest.osm.pbf',
  'IMPORT_STATUS': 'IN_PROGRESS',
  'IMPORT_TS': Timestamp('2025-12-03 14:17:36.017029')},
 {'PBF_NAME': 'denmark-latest.osm.pbf',
  'IMPORT_STATUS': 'FAILED',
  'IMPORT_TS': Timestamp('2025-12-06 06:26:45.951064')},
 {'PBF_NAME': 'lithuania-latest.osm.pbf',
  'IMPORT_STATUS': 'FAILED',
  'IMPORT_TS': Timestamp('2025-12-

In [2]:
def parse_country_from_importname(name: str) -> str:
    """
    Extract country name from PBF import name.
    
    Examples:
      andorra-latest.osm.pbf      -> andorra
      isle-of-man-251117.osm.pbf  -> isle-of-man
      latvia-latest.osm.pbf       -> latvia
      netherlands-latest.osm.pbf  -> netherlands
    """
    # Remove file extensions
    base = name
    if base.endswith(".osm.pbf"):
        base = base[:-8]
    elif base.endswith(".pbf"):
        base = base[:-4]
    
    # Split by hyphen and remove date/version suffixes
    parts = base.split("-")
    
    # Filter out 'latest' and date-like patterns (6-digit numbers)
    country_parts = []
    for part in parts:
        if part == "latest":
            continue
        if part.isdigit() and len(part) == 6:  # Date pattern like 251117
            continue
        country_parts.append(part)
    
    # Join remaining parts with hyphen
    if country_parts:
        return "-".join(country_parts)
    
    return base

In [3]:
def extract_date_from_import(row) -> str:
    """
    Extract date from PBF import record.
    
    If filename contains a date pattern (YYMMDD), convert to YYYY-MM-DD.
    If filename contains 'latest', use the IMPORT_TS date.
    
    Examples:
      isle-of-man-251117.osm.pbf -> 2025-11-17
      latvia-latest.osm.pbf with IMPORT_TS 2025-12-03 -> 2025-12-03
    """
    import pandas as pd
    from datetime import datetime
    
    name = row['PBF_NAME']
    import_ts = row['IMPORT_TS']
    
    # Remove file extensions
    base = name
    if base.endswith(".osm.pbf"):
        base = base[:-8]
    elif base.endswith(".pbf"):
        base = base[:-4]
    
    # Split by hyphen to find date pattern
    parts = base.split("-")
    
    # Look for 6-digit date pattern (YYMMDD)
    for part in parts:
        if part.isdigit() and len(part) == 6:
            # Parse as YYMMDD
            yy = part[:2]
            mm = part[2:4]
            dd = part[4:6]
            # Assume 20xx for year
            yyyy = f"20{yy}"
            return f"{yyyy}-{mm}-{dd}"
    
    # If 'latest' is in filename, use IMPORT_TS
    if 'latest' in parts:
        if isinstance(import_ts, pd.Timestamp):
            return import_ts.strftime('%Y-%m-%d')
        elif import_ts:
            return str(import_ts)[:10]
    
    # Fallback: use IMPORT_TS
    if isinstance(import_ts, pd.Timestamp):
        return import_ts.strftime('%Y-%m-%d')
    elif import_ts:
        return str(import_ts)[:10]
    
    return None

In [4]:
pbf_df['COUNTRY'] = pbf_df['PBF_NAME'].apply(parse_country_from_importname)
pbf_df['UP_TO_DATE'] = pbf_df.apply(extract_date_from_import, axis=1)
pbf_df

,PBF_NAME,IMPORT_STATUS,IMPORT_TS,COUNTRY,UP_TO_DATE
0,andorra-latest.osm.pbf,FAILED,2025-12-03 10:59:08.238218,andorra,2025-12-03
1,isle-of-man-251117.osm.pbf,FAILED,2025-12-03 11:00:29.752237,isle-of-man,2025-11-17
2,latvia-latest.osm.pbf,FAILED,2025-12-03 11:31:56.206416,latvia,2025-12-03
3,albania-251117.osm.pbf,IN_PROGRESS,2025-12-03 12:42:21.821054,albania,2025-11-17
4,estonia-251127.osm.pbf,FAILED,2025-12-03 13:01:19.085699,estonia,2025-11-27
5,belarus-latest.osm.pbf,IN_PROGRESS,2025-12-03 14:17:36.017029,belarus,2025-12-03
6,denmark-latest.osm.pbf,FAILED,2025-12-06 06:26:45.951064,denmark,2025-12-06
7,lithuania-latest.osm.pbf,FAILED,2025-12-06 06:30:40.332389,lithuania,2025-12-06
8,bulgaria-latest.osm.pbf,FAILED,2025-12-06 07:45:35.901058,bulgaria,2025-12-06
9,netherlands-latest.osm.pbf,COMPLETE,2025-12-06 09:34:25.909771,netherlands,2025-12-06


In [5]:
pbf_df[["COUNTRY", "UP_TO_DATE"]].to_dict(orient='records')

[{'COUNTRY': 'andorra', 'UP_TO_DATE': '2025-12-03'},
 {'COUNTRY': 'isle-of-man', 'UP_TO_DATE': '2025-11-17'},
 {'COUNTRY': 'latvia', 'UP_TO_DATE': '2025-12-03'},
 {'COUNTRY': 'albania', 'UP_TO_DATE': '2025-11-17'},
 {'COUNTRY': 'estonia', 'UP_TO_DATE': '2025-11-27'},
 {'COUNTRY': 'belarus', 'UP_TO_DATE': '2025-12-03'},
 {'COUNTRY': 'denmark', 'UP_TO_DATE': '2025-12-06'},
 {'COUNTRY': 'lithuania', 'UP_TO_DATE': '2025-12-06'},
 {'COUNTRY': 'bulgaria', 'UP_TO_DATE': '2025-12-06'},
 {'COUNTRY': 'netherlands', 'UP_TO_DATE': '2025-12-06'}]

In [6]:
def get_country_dates_from_pbf_imports():
    """
    Query PBF imports from HANA and extract country names and dates.
    
    Returns:
        list: List of dictionaries with 'COUNTRY' and 'UP_TO_DATE' keys
    """
    import os
    from dotenv import load_dotenv
    load_dotenv()
    from hdbcli import dbapi
    
    # HANA Credentials
    address = os.getenv("HANA_HOST")
    port = os.getenv("HANA_PORT")
    user = os.getenv("HANA_USER")
    password = os.getenv("HANA_PASSWORD")
    
    connection = dbapi.connect(address=address, port=port, user=user, password=password)
    cursor = connection.cursor()
    
    pbf_query = """
    SELECT PBF_NAME, IMPORT_STATUS, IMPORT_TS FROM "GEO_FAB"."PBF_IMPORTS";
    """
    
    cursor.execute(pbf_query)
    pbf_rows = cursor.fetchall()
    
    cursor.close()
    connection.close()
    
    # Process without pandas
    result = []
    for row in pbf_rows:
        pbf_name = row[0]
        import_status = row[1]
        import_ts = row[2]
        
        # Extract country
        base = pbf_name
        if base.endswith(".osm.pbf"):
            base = base[:-8]
        elif base.endswith(".pbf"):
            base = base[:-4]
        
        parts = base.split("-")
        country_parts = []
        for part in parts:
            if part == "latest":
                continue
            if part.isdigit() and len(part) == 6:
                continue
            country_parts.append(part)
        
        country = "-".join(country_parts) if country_parts else base
        
        # Extract date
        up_to_date = None
        for part in parts:
            if part.isdigit() and len(part) == 6:
                yy = part[:2]
                mm = part[2:4]
                dd = part[4:6]
                up_to_date = f"20{yy}-{mm}-{dd}"
                break
        
        if up_to_date is None and 'latest' in parts:
            if hasattr(import_ts, 'strftime'):
                up_to_date = import_ts.strftime('%Y-%m-%d')
            elif import_ts:
                up_to_date = str(import_ts)[:10]
        
        if up_to_date is None:
            if hasattr(import_ts, 'strftime'):
                up_to_date = import_ts.strftime('%Y-%m-%d')
            elif import_ts:
                up_to_date = str(import_ts)[:10]
        
        result.append({'COUNTRY': country, 'UP_TO_DATE': up_to_date})
    
    return result

# Call the function
get_country_dates_from_pbf_imports()

[{'COUNTRY': 'andorra', 'UP_TO_DATE': '2025-12-03'},
 {'COUNTRY': 'isle-of-man', 'UP_TO_DATE': '2025-11-17'},
 {'COUNTRY': 'latvia', 'UP_TO_DATE': '2025-12-03'},
 {'COUNTRY': 'albania', 'UP_TO_DATE': '2025-11-17'},
 {'COUNTRY': 'estonia', 'UP_TO_DATE': '2025-11-27'},
 {'COUNTRY': 'belarus', 'UP_TO_DATE': '2025-12-03'},
 {'COUNTRY': 'denmark', 'UP_TO_DATE': '2025-12-06'},
 {'COUNTRY': 'lithuania', 'UP_TO_DATE': '2025-12-06'},
 {'COUNTRY': 'bulgaria', 'UP_TO_DATE': '2025-12-06'},
 {'COUNTRY': 'netherlands', 'UP_TO_DATE': '2025-12-06'}]